# Занятие 34. Практика: bagging и случайный лес

Вы **пишете код и текст сами** в пустых ячейках после каждого задания. Блоки **«Легенда»** и **«Дано»** (и сломанный код в «Дано», если он есть) не меняйте.

Вы исследуете **ансамбль деревьев** не как «гонку за accuracy», а как **абляцию**:
какие ингредиенты (bootstrap, random features, число деревьев) реально дают прирост и стабильность.

Теория — занятие 33, ноутбук `Урок_33_Ансамбли_Bagging_Случайный_лес.ipynb`.
Главная модель: **RandomForestClassifier** (+ сравнение с одним деревом и bagging).

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 1 | Импорты и split журнала контактов | 2 |
| 2 | Кодовая абляция: дерево / bagging / RF / n_estimators / bootstrap | 5 |
| 3 | Gradio-пульт абляции | 4 |
| 4 | Протокол экспериментов (`experiments_log`) | 5 |
| 5 | OOB: контакты, которые смена не видела | 3 |
| 6 | Детектив: permutation importance приборов | 4 |
| 7 | Детектив: странный контакт и отключение признаков | 5 |
| 8 | Итоговые выводы | 2 |
| | **Итого** | **30** |

**Часть A — абляция + пульт + протокол** (задания 1–4).
**Часть B — детектив по приборам** (задания 5–7).


---
## Легенда: центр сопровождения «Orbital Yard»

Вы — аналитик в центре **Orbital Yard**. По ночному небу летят контакты трёх типов:

| Класс | Что это |
|-------|---------|
| `satellite` | рабочий спутник |
| `debris` | обломок / мусор |
| `glitch` | ложное срабатывание приборов |

По каждому контакту пишут показания приборов: `radar_rcs`, `optical_mag`, `ir_delta`,
`doppler_shift`, `spin_period`, а также четыре «шумовых» канала `noise_0`…`noise_3`
(калибровочный мусор, который в журнал попал по ошибке).

Ваша смена делает две вещи:

1. **Абляционный пульт** — включает/выключает ингредиенты леса и ведёт **протокол экспериментов**.
2. **Детектив по приборам** — выясняет, какие датчики реально помогают, а какие — пустышки.


---
## Дано: журнал контактов

Ячейку ниже **не меняйте**. Она создаёт синтетический журнал Orbital Yard
с именованными приборами и шумовыми каналами `noise_*`.

После запуска будут:
- `df` — таблица контактов;
- `X`, `y` — признаки и метки (`0=satellite`, `1=debris`, `2=glitch`);
- `FEATURE_NAMES`, `CLASS_NAMES`.

> Если позже не импортируется Gradio: `pip install gradio` (или `!pip install gradio` в ячейке).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 900

# Полезные приборы центра Orbital Yard
radar_rcs = rng.normal(0, 1, n)          # радиолокационный RCS
optical_mag = rng.normal(0, 1, n)        # оптическая яркость
ir_delta = rng.normal(0, 1, n)           # ИК-сигнатура
doppler_shift = rng.normal(0, 1, n)      # доплеровский сдвиг
spin_period = rng.normal(0, 1, n)        # период вращения

# Три класса контактов: satellite / debris / glitch
score_sat = 1.6 * radar_rcs - 1.2 * optical_mag + 0.8 * doppler_shift
score_deb = -1.1 * radar_rcs + 1.8 * ir_delta + 1.0 * spin_period
score_gli = 0.3 * radar_rcs + 0.4 * optical_mag - 1.7 * ir_delta + 1.5 * doppler_shift
logits = np.column_stack([score_sat, score_deb, score_gli])
ex = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = ex / ex.sum(axis=1, keepdims=True)
y = np.array([rng.choice(3, p=p) for p in probs])

# Небольшой шум меток (ошибки операторов)
flip = rng.random(n) < 0.06
y[flip] = rng.integers(0, 3, size=int(flip.sum()))

# Шумовые каналы приборов — не несут сигнала о классе
noise = rng.normal(0, 1, size=(n, 4))

FEATURE_NAMES = [
    "radar_rcs",
    "optical_mag",
    "ir_delta",
    "doppler_shift",
    "spin_period",
    "noise_0",
    "noise_1",
    "noise_2",
    "noise_3",
]
CLASS_NAMES = ["satellite", "debris", "glitch"]

X = np.column_stack([radar_rcs, optical_mag, ir_delta, doppler_shift, spin_period, noise])
df = pd.DataFrame(X, columns=FEATURE_NAMES)
df["contact_type"] = [CLASS_NAMES[i] for i in y]

print("Журнал контактов Orbital Yard:")
print(df.head())
print()
print("Размер:", df.shape)
print("Классы:")
print(df["contact_type"].value_counts())


---
## Задание 1. Импорты и split — **2 балла**

Подготовьте лабораторию абляции.

**Шаг 1.** Импортируйте:
`train_test_split`, `DecisionTreeClassifier`, `BaggingClassifier`,
`RandomForestClassifier`, `accuracy_score`, `permutation_importance`.

**Шаг 2.** Задайте `RANDOM_STATE = 42`.

**Шаг 3.** Разделите `X`, `y` на train / validation (**70 / 30**),
`stratify=y`, `random_state=RANDOM_STATE`.
Сохраните `X_train`, `X_val`, `y_train`, `y_val`.

**Шаг 4.** Выведите размеры выборок и доли классов.

### Подробные критерии (для проверки LLM)

- **0.5 балла** — импортированы нужные классы/функции.
- **0.5 балла** — задан `RANDOM_STATE = 42`.
- **0.5 балла** — split 70/30 со `stratify=y`.
- **0.5 балла** — выведены размеры и/или доли классов.

### Снижение баллов

- Нет `stratify` → минус **0.5**.
- Validation используется до обучения как «вторая train» → минус **1.0**.


---
## Задание 2. Кодовая абляция — **5 баллов**

Соберите **таблицу абляции** одного семейства моделей. Это не лидерборд «кто круче»,
а ответ на вопрос: **что именно** даёт прирост?

Обучите и сравните на **одном и том же** validation:

1. одно `DecisionTreeClassifier`;
2. `BaggingClassifier` из деревьев (`n_estimators=150`) — bootstrap, **без** random features;
3. `RandomForestClassifier` (`n_estimators=150`, `max_features="sqrt"`);
4. RF с малым и большим `n_estimators` (например 10 и 300);
5. RF с `bootstrap=False` (если API позволяет).

Для каждой строки сохраните train/validation accuracy.
Постройте **bar**-график конфигураций и **line**-график `n_estimators → val accuracy`.

Графики: заголовок, подписи осей, легенда где нужна.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучено одно дерево и посчитаны train/val accuracy.
- **1.0 балл** — обучен bagging (bootstrap, без random features) и сравнён с деревом.
- **1.0 балл** — обучен RF с `max_features="sqrt"` и сравнён с bagging.
- **1.0 балл** — сравнены малый и большой `n_estimators` у RF (+ опционально `bootstrap=False`).
- **1.0 балл** — есть таблица абляции и bar/line-графики с заголовком и подписями осей.

### Снижение баллов

- Сравнивают только train accuracy → минус **1.5**.
- Нет графика → минус **1.0**.
- Разные `random_state`/разные split между моделями без фиксации → минус **0.5**.


---
## Задание 3. Gradio-пульт абляции — **4 балла**

Соберите **интерактивный веб-пульт** (предпочтительно **Gradio** `gr.Blocks`).

Тумблеры / контроли:

- режим: `одно дерево` / `bagging` / `random forest`;
- слайдер `n_estimators`;
- `max_depth` (None или число);
- `max_features` для RF: `sqrt` / `log2` / `None` / доля;
- `bootstrap` on/off (для bagging/RF);
- кнопка **«Пересчитать»**;
- кнопка **«Добавить в протокол»**.

Вывод пульта: текст «что включено», train/val accuracy, короткий bar-график.
Запуск: `demo.launch(share=False)` — выполните ячейку, откроется **локальный** интерфейс.

Если `import gradio` не работает: `pip install gradio`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — есть Gradio UI (`gr.Blocks` / Interface) с режимом модели.
- **1.0 балл** — есть слайдеры/контроли `n_estimators`, `max_depth`, `max_features`, `bootstrap`.
- **1.0 балл** — кнопка пересчёта обучает модель на train и показывает train/val accuracy.
- **1.0 балл** — UI запускается через `demo.launch(share=False)` (или эквивалент) из ячейки.

### Снижение баллов

- Нет интерактивных контролов (только статичный код) → минус **2.0**.
- Модель учится с подглядыванием в validation при `fit` → минус **1.0**.
- Fallback на `ipywidgets` допустим, если Gradio недоступен, но контроли должны быть.


---
## Задание 4. Протокол экспериментов — **5 баллов**

Ведите **лабораторный протокол** `experiments_log` (DataFrame и/или markdown-таблица).

Каждая строка — один эксперимент:

| Поле | Смысл |
|------|--------|
| гипотеза | что проверяете тумблерами |
| режим / n_estimators / max_depth / max_features / bootstrap | настройки пульта |
| train_acc / val_acc | метрики |
| вывод | что показали числа |

**Минимум 5 строк**, и среди них должны быть **разные** конфигурации
(дерево vs bagging vs RF, разный `n_estimators`, желательно bootstrap on/off).

Используйте пульт (кнопка «Добавить в протокол») или заполните таблицу кодом после серии запусков.
В конце выведите итоговый `experiments_log` и коротко ответьте: **какой ингредиент дал главный прирост?**

### Подробные критерии (для проверки LLM)

- **1.0 балл** — есть таблица/DataFrame протокола с полями настроек и метрик.
- **1.5 балла** — не меньше **5** строк экспериментов.
- **1.0 балл** — конфигурации реально разные (не копипаста одной строки).
- **1.0 балл** — у каждой (или почти каждой) строки есть краткий вывод/гипотеза.
- **0.5 балла** — итоговый вывод: что дало основной прирост (обычно bagging vs одно дерево).

### Снижение баллов

- Меньше 5 экспериментов → минус **1.5**.
- В протоколе только train без validation → минус **1.0**.
- Выводы не связаны с числами протокола → минус **0.5**.


*(Ваш ответ)*


---
## Задание 5. OOB: контакты вне смены — **3 балла**

Обучите `RandomForestClassifier(..., oob_score=True)` на **train**.
Сравните `oob_score_` с validation accuracy.

Смысл OOB простыми словами: для каждого дерева есть контакты,
**которые эта смена (bootstrap-мешок) не видела**. Их ответы — честная быстрая проверка.
Это **не** финальный скрытый test всего проекта.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — RF обучен с `oob_score=True` на train.
- **1.0 балл** — выведены OOB score и validation accuracy.
- **1.0 балл** — в markdown/print есть пояснение, что OOB ≠ финальный test.

### Снижение баллов

- OOB считают на validation напрямую «вручную» вместо `oob_score_` → минус **0.5**
  (если идея верная, но API не использован).
- Путают OOB с test и предлагают подбирать всё только по OOB без оговорок → минус **1.0**.


---
## Задание 6. Детектив: permutation importance — **4 балла**

Обучите RF на train. На **validation** посчитайте `permutation_importance`.

Покажите таблицу и barh-график. Ответьте:

1. какие приборы реально полезны;
2. что каналы `noise_*` бесполезны (важность около нуля).

### Подробные критерии (для проверки LLM)

- **1.0 балл** — RF обучен на train.
- **1.5 балла** — `permutation_importance` посчитан на validation.
- **0.5 балла** — есть таблица/сортировка важностей.
- **1.0 балл** — явно отмечены полезные приборы и бесполезность `noise_*`.

### Снижение баллов

- Importance считают на train и выдают за «боевую» полезность без оговорки → минус **0.5**.
- Нет вывода про `noise_*` → минус **1.0**.


---
## Задание 7. Детектив: странный контакт — **5 баллов**

Возьмите (или сконструируйте) **странный контакт**: конфликт показаний приборов
и большой `noise_*`.

1. Получите `predict_proba` обученного RF.
2. По очереди «выключайте» признаки (замена на медиану train) и смотрите,
   как падает вероятность исходного класса — какой прибор сильнее тянет решение.
3. Дополнительно сравните validation accuracy при отключении групп:
   без `noise_*`, без ключевого прибора, только на `noise_*`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — построен/выбран странный контакт и получен `predict_proba`.
- **1.5 балла** — есть анализ влияния признаков через замену значения / importance на кейсе.
- **1.5 балла** — есть абляция групп признаков (noise / ключевой прибор) с метрикой на validation.
- **1.0 балл** — сформулирован вывод: какой признак тянет кейс; `noise_*` не тянут.

### Снижение баллов

- Нет работы с вероятностями/`predict_proba` → минус **1.0**.
- Вывод противоречит числам (например, объявляют `noise_*` главными) → минус **1.5**.


---
## Задание 8. Итоговые выводы — **2 балла**

Напишите **три** коротких вывода по своим числам и протоколу:

1. какой ингредиент абляции дал главный прирост;
2. зачем нужен протокол экспериментов (а не одна цифра accuracy);
3. что показал детектив по приборам (`noise_*` vs реальные датчики).

### Подробные критерии (для проверки LLM)

- **0.7 балла** — вывод про главный ингредиент абляции.
- **0.6 балла** — вывод про ценность протокола.
- **0.7 балла** — вывод про приборы / `noise_*`.

### Снижение баллов

- Общие фразы без опоры на таблицу/протокол → минус **0.5**.


*(Ваш ответ)*
